# The Hidden Precision Trap of bfloat16 Training
Mixed precision training often seems like a free performance boost. Since bfloat16 uses the same 8-bit exponent as float32, it can represent numbers across the same range while using half the memory. This leads many people to assume that simply converting everything to bfloat16 is safe. However, while the exponent determines how large or small numbers can be, the mantissa determines how precisely they can be represented. bfloat16 reduces the mantissa from 23 bits to just 7 bits, significantly lowering numerical precision.

This loss of precision becomes important late in training when gradients become very small. At that stage, many gradient values fall between the discrete values that bfloat16 can represent and are rounded to zero. Once this happens, the optimizer stops receiving useful update signals and some parameters stop changing, causing learning to stall. In the example below, we use a simple NumPy simulation to compare full bfloat16, mixed precision, and float32 training, showing exactly when gradients disappear and how the resulting convergence behavior begins to diverge.

## Installing the dependencies

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

## Bit-layout constants
This first cell establishes the numerical foundations of the experiment by comparing the internal bit layouts of float32, bfloat16, and float16. Although bfloat16 is often considered a safe replacement for float32 because both formats share the same 8-bit exponent and therefore the same dynamic range, this comparison highlights the critical difference in precision: the mantissa. By reducing the mantissa from 23 bits to just 7 bits, bfloat16 can represent far fewer distinct values between neighboring numbers.

The printed precision gaps make this concrete, showing that gradients as small as 1e-7 can fall below the resolution of bfloat16 and be rounded away entirely. Understanding this trade-off between range and precision is essential before we simulate training, because the rest of the notebook demonstrates how these seemingly tiny rounding effects can accumulate and eventually cause learning to stall.

In [5]:
print("=" * 62)
print("  FORMAT COMPARISON")
print("=" * 62)
print(f"  {'Format':<12} {'Exponent bits':>14} {'Mantissa bits':>14}")
print("-" * 62)
print(f"  {'float32':<12} {'8':>14} {'23':>14}")
print(f"  {'bfloat16':<12} {'8':>14} {'7':>14}")
print(f"  {'float16':<12} {'5':>14} {'10':>14}")
print("=" * 62)
print()
print("  bfloat16 precision gap (mantissa):  2^-7  ≈  0.0078")
print("  float32  precision gap (mantissa):  2^-23 ≈  0.000000119")
print()
print("  → A gradient of 1e-7 is ~840× smaller than bfloat16's")
print("    finest representable step near 1.0. It rounds to ZERO.")
print()

  FORMAT COMPARISON
  Format        Exponent bits  Mantissa bits
--------------------------------------------------------------
  float32                   8             23
  bfloat16                  8              7
  float16                   5             10

  bfloat16 precision gap (mantissa):  2^-7  ≈  0.0078
  float32  precision gap (mantissa):  2^-23 ≈  0.000000119

  → A gradient of 1e-7 is ~840× smaller than bfloat16's
    finest representable step near 1.0. It rounds to ZERO.



## Representable value density
This cell visualizes what "lower precision" actually means in practice. Instead of discussing mantissa bits abstractly, we count how many distinct numbers each format can represent within the narrow interval [1.0, 1.01]. The result is striking: float32 can represent nearly 84,000 unique values in this range, while the bfloat16 approximation can represent only 11. In other words, float32 provides thousands of times finer numerical resolution.

Every gradient update must ultimately be mapped onto one of these representable values, so a coarse grid means many small updates cannot be represented accurately. When gradients become sufficiently small, they fall between neighboring bfloat16 values and are rounded away, causing the optimizer to lose information that would otherwise continue nudging the model toward a better solution.

In [6]:
def count_representable(low, high, dtype):
    """Step through floats in [low, high] for a given dtype."""
    val = np.array([low], dtype=dtype)
    count = 0
    while float(val[0]) <= high:
        count += 1
        # next representable value: add the smallest possible step
        val = np.nextafter(val, np.array([high * 10], dtype=dtype))
    return count

range_low, range_high = 1.0, 1.01

n_f32  = count_representable(range_low, range_high, np.float32)
n_bf16 = count_representable(range_low, range_high, np.float16)  # approximation

print("=" * 62)
print("  REPRESENTABLE VALUES IN [1.000, 1.010]")
print("=" * 62)
print(f"  float32   : {n_f32:>10,} distinct values")
print(f"  bfloat16* : {n_bf16:>10,} distinct values  (* via float16 mantissa proxy)")
print()
print(f"  Ratio     : float32 is ~{n_f32 // max(n_bf16,1):,}× finer in this range")
print()
print("  A gradient sitting between two bfloat16 grid points")
print("  gets rounded to the nearest one — possibly zero.")
print()

  REPRESENTABLE VALUES IN [1.000, 1.010]
  float32   :     83,887 distinct values
  bfloat16* :         11 distinct values  (* via float16 mantissa proxy)

  Ratio     : float32 is ~7,626× finer in this range

  A gradient sitting between two bfloat16 grid points
  gets rounded to the nearest one — possibly zero.



## The rounding-to-zero demonstration
This cell demonstrates the core failure mode directly. We start with a weight near 1.0 and apply a very small gradient that would naturally appear during the later stages of training. In the full-precision path, the gradient remains intact, producing a tiny but valid parameter update. In the low-precision path, however, the gradient is first cast to bfloat16 (approximated here with float16), where it rounds to zero because it falls below the format's representable resolution. As a result, the optimizer computes a zero update and the weight remains unchanged. The numerical difference after a single step is minuscule—only 5×10^−11 — but the significance lies in the accumulation of these events. When thousands or millions of gradients are rounded away during fine-tuning, parameters gradually stop moving, causing training to plateau even though useful learning signals still exist in the data.

In [7]:
weight_f32  = np.float64(1.0)   # use float64 so tiny updates are visible
weight_bf16 = np.float64(1.0)

lr   = np.float64(0.01)
grad = np.float64(5e-9)          # tiny but legitimate gradient — below bf16 floor

# float32 path — full precision accumulation
update_f32     = lr * grad
weight_f32_new = weight_f32 - update_f32

# bfloat16 path — gradient cast to low precision (float16 as proxy)
grad_bf16       = np.float16(grad)        # rounds to 0.0
update_bf16     = lr * float(grad_bf16)
weight_bf16_new = weight_bf16 - update_bf16

zero_label = "← rounds to ZERO ✓" if grad_bf16 == 0.0 else f"← {grad_bf16:.2e} (imprecise)"

print("=" * 62)
print("  THE ROUNDING-TO-ZERO DEMONSTRATION")
print("=" * 62)
print(f"  Original weight          : {weight_f32:.2e}")
print(f"  Learning rate            : {lr:.4f}")
print(f"  True gradient (float32)  : {grad:.2e}")
print()
print(f"  gradient in bfloat16     : {grad_bf16:.2e}  {zero_label}")
print()
print(f"  float32  update applied  : -{update_f32:.2e}")
print(f"  bfloat16 update applied  : -{update_bf16:.2e}")
print()
print(f"  New weight (float32  path) : {weight_f32_new:.15f}")
print(f"  New weight (bfloat16 path) : {weight_bf16_new:.15f}")
print()
diff = abs(weight_f32_new - weight_bf16_new)
print(f"  Difference between paths   : {diff:.2e}")
print()
print("  float32 applied a real update; bfloat16 applied zero.")
print("  Across thousands of late-stage steps, bfloat16 stops")
print("  making progress entirely — the parameter is frozen.")
print()

  THE ROUNDING-TO-ZERO DEMONSTRATION
  Original weight          : 1.00e+00
  Learning rate            : 0.0100
  True gradient (float32)  : 5.00e-09

  gradient in bfloat16     : 0.00e+00  ← rounds to ZERO ✓

  float32  update applied  : -5.00e-11
  bfloat16 update applied  : -0.00e+00

  New weight (float32  path) : 0.999999999950000
  New weight (bfloat16 path) : 1.000000000000000

  Difference between paths   : 5.00e-11

  float32 applied a real update; bfloat16 applied zero.
  Across thousands of late-stage steps, bfloat16 stops
  making progress entirely — the parameter is frozen.



## Simulated training loop
This cell moves from isolated examples to a complete training simulation. Using the simple quadratic objective L(w)=(w−0)^2, we compare three common precision strategies: full low-precision training, mixed precision training, and full float32 training. As the weight approaches the optimum, the gradient naturally becomes smaller, mimicking what happens during the final stages of neural network training. This is precisely where precision limitations matter most. The results show that full low-precision training converges more slowly and stalls at a larger residual error because many of the smallest updates are lost to rounding. In contrast, mixed precision closely matches full float32 performance because the optimizer maintains a high-precision master copy of the weights, allowing tiny updates to accumulate even when the forward and backward passes run in reduced precision. This design is why modern training frameworks use mixed precision rather than storing everything in bfloat16.

In [8]:
TARGET   = 0.0          # optimal weight value
INIT_W   = 2.0          # starting point
LR       = 0.05
STEPS    = 500

def simulate(strategy: str):
    """
    strategy: 'full_bf16' | 'mixed' | 'full_fp32'
    Returns list of (step, loss, weight) tuples.
    """
    w = INIT_W
    history = []

    for step in range(STEPS):
        loss = (w - TARGET) ** 2
        grad = 2.0 * (w - TARGET)   # true fp64 gradient

        if strategy == "full_bf16":
            # cast both weight and gradient to low precision
            w_low   = np.float16(w)
            g_low   = np.float16(grad)
            w       = float(w_low) - LR * float(g_low)

        elif strategy == "mixed":
            # compute gradient in low precision (efficiency)
            g_low   = np.float16(grad)
            # accumulate update in fp32 master weight
            w       = w - LR * float(g_low)   # w itself stays fp64/fp32

        else:  # full_fp32
            w       = w - LR * grad

        history.append((step, loss, w))

    return history

hist_bf16  = simulate("full_bf16")
hist_mixed = simulate("mixed")
hist_fp32  = simulate("full_fp32")

# Extract losses
losses_bf16  = [h[1] for h in hist_bf16]
losses_mixed = [h[1] for h in hist_mixed]
losses_fp32  = [h[1] for h in hist_fp32]

final_w_bf16  = hist_bf16[-1][2]
final_w_mixed = hist_mixed[-1][2]
final_w_fp32  = hist_fp32[-1][2]

print("=" * 62)
print("  SIMULATED TRAINING — 500 STEPS, L(w) = (w - 0)²")
print("=" * 62)
print(f"  {'Strategy':<22} {'Final Loss':>12} {'Final Weight':>14}")
print("-" * 62)
print(f"  {'Full bfloat16':<22} {losses_bf16[-1]:>12.6f} {final_w_bf16:>14.8f}")
print(f"  {'Mixed precision':<22} {losses_mixed[-1]:>12.6f} {final_w_mixed:>14.8f}")
print(f"  {'Full float32':<22} {losses_fp32[-1]:>12.6f} {final_w_fp32:>14.8f}")
print()
print("  Full bfloat16 stalls because tiny late-stage gradients")
print("  round to zero. Mixed precision converges like float32.")
print()

  SIMULATED TRAINING — 500 STEPS, L(w) = (w - 0)²
  Strategy                 Final Loss   Final Weight
--------------------------------------------------------------
  Full bfloat16              0.000000     0.00000022
  Mixed precision            0.000000     0.00000001
  Full float32               0.000000     0.00000000

  Full bfloat16 stalls because tiny late-stage gradients
  round to zero. Mixed precision converges like float32.



## Gradient magnitude survival analysis
This cell quantifies exactly which gradient magnitudes survive the transition to low precision and which ones disappear. By sweeping gradients across eight orders of magnitude, from 10^−9 to 10^−1, and measuring how much of each value remains after conversion to the lower-precision format, we can identify the practical limits of representability. The results show that larger gradients are preserved almost perfectly, while gradients around 10^−8 are effectively annihilated by rounding.

This explains why mixed precision training usually works well during the early stages of optimization—when gradients are relatively large—but can encounter difficulties late in training, when updates become extremely small. The key takeaway is that the problem is not a gradual loss of accuracy; below a certain threshold, gradients stop existing altogether from the optimizer's perspective, turning what should be a small update into no update at all.

In [9]:
grad_magnitudes = np.logspace(-9, -1, 300)
survival        = []

for g in grad_magnitudes:
    g_low    = float(np.float16(g))
    retained = g_low / g if g > 0 else 0.0
    survival.append(retained)

survival = np.array(survival)

print("=" * 62)
print("  GRADIENT SURVIVAL THROUGH BFLOAT16 (proxy)")
print("=" * 62)
print(f"  Gradient ≥ 1e-4  : ~{survival[grad_magnitudes >= 1e-4].mean():.1%} retained on average")
print(f"  Gradient ~ 1e-6  : ~{survival[np.isclose(grad_magnitudes, 1e-6, rtol=0.5)].mean():.1%} retained")
print(f"  Gradient ~ 1e-8  : ~{survival[np.isclose(grad_magnitudes, 1e-8, rtol=0.5)].mean():.1%} retained")
print()
print("  Gradients below ~1e-7 are almost entirely lost in bf16.")
print()

  GRADIENT SURVIVAL THROUGH BFLOAT16 (proxy)
  Gradient ≥ 1e-4  : ~100.0% retained on average
  Gradient ~ 1e-6  : ~100.6% retained
  Gradient ~ 1e-8  : ~0.0% retained

  Gradients below ~1e-7 are almost entirely lost in bf16.



## Summary
This final cell summarizes the central lesson of mixed precision training: not everything should be stored in low precision. The performance gains come from using bfloat16 for the computationally expensive forward and backward passes, where its large exponent range is usually sufficient and memory savings are substantial. However, the quantities responsible for accumulating information across training steps—master weights, optimizer momentum, optimizer variance, loss reductions, and matrix multiplication accumulators—must remain in float32.

These components repeatedly aggregate tiny numerical contributions, making them particularly vulnerable to the precision limitations of bfloat16. Modern deep learning frameworks therefore use a hybrid approach: low precision where speed matters most, and full precision where numerical accuracy is critical. This balance delivers most of the memory and throughput benefits of bfloat16 while avoiding the silent training failures demonstrated throughout the notebook.

In [12]:
print("=" * 62)
print("  MIXED PRECISION — WHAT STAYS IN FLOAT32")
print("=" * 62)
components = [
    ("Forward activations",    "bfloat16", "Speed & memory; range is sufficient"),
    ("Backward gradients",     "bfloat16", "Large-magnitude signals survive"),
    ("Master weights",         "float32 ⚠", "Must accumulate tiny updates"),
    ("Optimizer momentum",     "float32 ⚠", "Running sum; precision compounds"),
    ("Optimizer variance",     "float32 ⚠", "AdaGrad/Adam state; same risk"),
    ("Loss scalar",            "float32 ⚠", "Avoid reduction overflow"),
    ("GEMM accumulators",      "float32 ⚠", "HW auto-promotes; never override"),
]
print(f"  {'Component':<28} {'Precision':<12} {'Reason'}")
print("-" * 62)
for comp, prec, reason in components:
    print(f"  {comp:<28} {prec:<12} {reason}")
print()
print("=" * 62)
print("  DEMO COMPLETE")
print("=" * 62)

  MIXED PRECISION — WHAT STAYS IN FLOAT32
  Component                    Precision    Reason
--------------------------------------------------------------
  Forward activations          bfloat16     Speed & memory; range is sufficient
  Backward gradients           bfloat16     Large-magnitude signals survive
  Master weights               float32 ⚠    Must accumulate tiny updates
  Optimizer momentum           float32 ⚠    Running sum; precision compounds
  Optimizer variance           float32 ⚠    AdaGrad/Adam state; same risk
  Loss scalar                  float32 ⚠    Avoid reduction overflow
  GEMM accumulators            float32 ⚠    HW auto-promotes; never override

  DEMO COMPLETE
